# 54 — Human Review & Audit

Materializa la cola Top-K para revisión humana y preserva la separación entre evidencia del modelo y decisiones humanas. No genera ninguna sanción o conclusión jurídica.

In [ ]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


In [ ]:
from argos.production.scoring import score_batch
from argos.production.review import BUILD_IMPLEMENTATION, build_review_queue
from argos.production.audit import verify_audit_chain

scored = score_batch(
    input_path=SHADOW_INPUT, root=paths.root, gold_spec=GOLD_SPEC, modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR, temp_directory=TEMP_DIR,
)
print("implementation:", BUILD_IMPLEMENTATION)
result = build_review_queue(
    scored_path=scored.scored_path,
    root=paths.root,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    temp_directory=TEMP_DIR,
)
result.queue_summary

In [ ]:
result.validation

In [ ]:
chain_ok, events, error = verify_audit_chain(paths.root, log_path=M6_SPEC.audit.log_path)
print({"audit_chain_ok": chain_ok, "events": events, "error": error})
assert chain_ok

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("54 ✅ PASS" if result.ready_for_next_notebook else "54 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Review queue:", result.queue_path)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook